<a href="https://colab.research.google.com/github/engelberger/frustrapy/blob/dev_plots/frustrapy_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Frustratometer in Python
<a href="https://colab.research.google.com/github/engelberger/frustrapy/blob/main/FrustraPy_colab.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Introduction

The concept of frustration in proteins refers to the presence of conflicting interactions within the protein structure. These conflicts arise when the local interactions within a protein are not optimally stabilizing, leading to a certain degree of energetic frustration. The following is a summary please refer to the original paper for more details:

[Protein Frustratometer 2: a tool to localize energetic frustration in protein molecules, now with electrostatics](https://academic.oup.com/nar/article/44/W1/W356/2499321)
[FrustrometerR: an R-package to compute local frustration in protein structures, point mutants and MD simulations](https://academic.oup.com/bioinformatics/article/37/18/3038/6171179)


## There are three main types of frustration

* *Highly frustrated*: Highly frustrated regions in a protein are those where the local interactions are significantly destabilizing compared to what would be expected in an idealized, energetically minimized structure. These regions often play crucial roles in protein function, such as binding sites, allosteric sites, or regions involved in conformational changes. For example, in an enzyme, the active site might be highly frustrated to allow for substrate binding and catalysis, which require a certain degree of flexibility and adaptability.
* *Neutral*: Neutral regions in a protein are those where the local interactions are neither significantly stabilizing nor destabilizing. These regions may not directly contribute to protein function but are essential for maintaining the overall structural integrity of the protein. Neutral regions can serve as a buffer between highly frustrated and minimally frustrated regions, allowing for the necessary flexibility and stability balance within the protein.
* *Minimally frustrated*: Minimally frustrated regions in a protein are those where the local interactions are highly optimized and stabilizing. These regions typically form the stable core of the protein and are essential for maintaining the native folded state. Minimally frustrated regions often consist of hydrophobic residues that pack tightly together, forming a stable foundation for the protein structure. For example, in the case of globular proteins, the hydrophobic core is usually minimally frustrated, contributing to the overall stability of the folded state.

## Significance of Frustration in Proteins:

* Protein folding: During the protein folding process, the polypeptide chain navigates through an energy landscape to reach its native state. The concept of minimal frustration suggests that evolution has optimized the folding landscape to minimize energetic conflicts, allowing proteins to fold efficiently and avoid getting trapped in non-native states.
* Allostery: Allosteric regulation in proteins often involves highly frustrated regions that undergo conformational changes upon ligand binding or other perturbations. These frustrated regions allow for the propagation of allosteric signals throughout the protein structure, enabling long-range communication and regulation of protein function.
* Protein-protein interactions: Protein interfaces often contain a mix of highly frustrated and minimally frustrated regions. The highly frustrated regions may contribute to the specificity and adaptability of the interaction, while the minimally frustrated regions provide stability to the complex. The balance between frustration and stability at the interface is crucial for the formation and regulation of protein complexes.

In [ ]:
# @title Install
%cd /content
%pip install -q biopython igraph leidenalg
%pip -q install git+https://github.com/engelberger/frustrapy.git@dev_plots
%pip install -q -U kaleido==0.2.1
%pip -q install py3dmol


/content
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# @title Frustratometer in Python {"display-mode":"form"}
mode = "configurational" # @param ["configurational", "singleresidue", "mutational"]
pdbs_dir = "/content" # @param {type:"string"}
results_dir = "/content/Results" # @param {type:"string"}
example = True # @param {type:"boolean"}
overwrite = False # @param {type:"boolean"}
debug = False # @param {type:"boolean"}

import sys
import frustrapy

# If the example is True, we will download the example files
if example:
    !wget -q http://www.rcsb.org/pdb/files/1fhj.pdb -O 1fhj.pdb
    !wget -q http://www.rcsb.org/pdb/files/2dn1.pdb -O 2dn1.pdb
    !wget -q http://www.rcsb.org/pdb/files/1m6k.pdb -O 1m6k.pdb

    pdbs_dir = "/content"
    results_dir = "/content/Results_example"
    # Remove any previous results
    !rm -rf /content/Results_example/*

if overwrite:
    if example:
        !rm -rf /content/Results/*
    else:
        import os
        # Convert the results_dir to an absolute path
        results_dir = os.path.abspath(results_dir)
        os.system(f"rm -rf {results_dir}/*")

plots_dir_dict = frustrapy.dir_frustration(
    pdbs_dir=pdbs_dir,
    mode=mode,
    results_dir=results_dir,
    debug=debug
)

### Plots

In [ ]:
import plotly.graph_objects as go

# Access the first element of the tuple, which is the dictionary of plots
plot_data_dict = plots_dir_dict[0]

# Iterate through the PDB IDs and their corresponding plots
for pdb_id, plot_types in plot_data_dict.items():
    for plot_type, fig in plot_types.items():
        # Wrap the figure using go.FigureWidget
        fig.show()

In [ ]:
import frustrapy
import os
import logging
logger = logging.getLogger(__name__)

logger.setLevel(logging.INFO)
from frustrapy.analysis.frustration_calculator import FrustrationCalculator

mode = "configurational"

# set the logger level to debug
pdb_file = "/content/2dn1.pdb"
pdb_id = None
chain = None
residues = None
electrostatics_k = 12
seq_dist = 12
graphics = True
visualization = True
results_dir = "/content/jupyter_results"
debug = True

In [ ]:

# Set flag for mutation calculations to suppress logging
is_mutation_calculation = mode == "singleresidue" and residues is not None
# Only log protocol for main calculations, not individual mutations
if is_mutation_calculation:
    logger.debug(f"\nRunning Frustration Protocol:")
    logger.debug(f"- Analysis Mode: {mode}")
    if pdb_file:
        logger.debug(f"- Input Structure: {os.path.basename(pdb_file)}")
    if chain:
        logger.debug(f"- Analyzing Chain(s): {chain}")
    if residues:
        for chain_id, res_list in residues.items():
            logger.debug(f"- Residues for Chain {chain_id}: {res_list}")
    logger.info(f"- Sequence Distance: {seq_dist}")
    if electrostatics_k is not None:
        logger.debug(f"- Electrostatics K: {electrostatics_k}")
    logger.debug(f"- Graphics Generation: {'Enabled' if graphics else 'Disabled'}")
    logger.debug(
        f"- Structure Visualization: {'Enabled' if visualization else 'Disabled'}\n"
    )


    logger.debug("Starting frustration calculation")
# Validate PDB file existence if provided
if pdb_file is not None:
    pdb_file = os.path.abspath(pdb_file)
    logger.debug(f"Using PDB file: {pdb_file}")
    if not os.path.exists(pdb_file):
        logger.error(f"PDB file not found: {pdb_file}")
        raise FileNotFoundError(f"PDB file not found: {pdb_file}")
# Make results_dir absolute path if provided
if results_dir is not None:
    results_dir = os.path.abspath(results_dir)
    logger.debug(f"Using results directory: {results_dir}")
logger.debug(f"Initializing FrustrationCalculator with mode: {mode}")



In [ ]:
calculator = FrustrationCalculator(
    pdb_file=pdb_file,
    pdb_id=pdb_id,
    chain=chain,
    residues=residues,
    electrostatics_k=electrostatics_k,
    seq_dist=seq_dist,
    mode=mode,
    graphics=graphics,
    visualization=visualization,
    results_dir=results_dir,
    debug=debug,
    is_mutation_calculation=is_mutation_calculation,
)
logger.debug("Starting calculation")
pdb_configurational, plots, density_results = calculator.calculate()
logger.debug("Calculation completed")

In [ ]:
from frustrapy.analysis.mutations import mutate_res_parallel, mutate_res
# set log level to debug
logger.setLevel(logging.INFO)
mutate_res_parallel(pdb=pdb_configurational, res_num=109, chain="A", split=True, method="threading")

Processing mutations for residue 109:   0%|          | 0/20 [00:00<?, ?it/s]

Processing mutations for residue 109:   5%|▌         | 1/20 [00:01<00:26,  1.40s/it]

Processing mutations for residue 109:  15%|█▌        | 3/20 [00:02<00:13,  1.25it/s, residue=109]

Processing mutations for residue 109:  25%|██▌       | 5/20 [00:05<00:17,  1.18s/it, residue=109]

Processing mutations for residue 109:  30%|███       | 6/20 [00:05<00:12,  1.10it/s, residue=109]

...

Processing mutations for residue 109:  65%|██████▌   | 13/20 [00:11<00:05,  1.30it/s, residue=109]

Processing mutations for residue 109:  75%|███████▌  | 15/20 [00:12<00:03,  1.43it/s, residue=109]

Processing mutations for residue 109:  85%|████████▌ | 17/20 [00:14<00:01,  1.53it/s, residue=109]

Processing mutations for residue 109:  95%|█████████▌| 19/20 [00:15<00:00,  1.59it/s, residue=109]

Processing mutations for residue 109: 100%|██████████| 20/20 [00:15<00:00,  1.31it/s, residue=109]


In [ ]:
from frustrapy.visualization import plot_mutate_res
fig = plot_mutate_res(pdb=pdb_configurational, res_num=109, chain="A", save=True)
# set plotly render to vscode from plotly io
import plotly.io as pio
pio.renderers.default = "colab"
fig.show()

In [ ]:
%load_ext autoreload
%autoreload 2


# Import the visualization function
from frustrapy.visualization.structure import view_config_contacts_py3dmol

# Visualize contacts for ALA_109-F using your existing pdb object
view_config_contacts_py3dmol(
    pdb=pdb_configurational,  # Your existing Pdb object
    central_chain="A",        # The chain of interest
    central_res=109,          # The residue number
    width=800,                # Optional: viewer width (default: 800)
    height=600                # Optional: viewer height (default: 600)
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Found 16 contacts to _109A


,Chain,Residue,State,Magnitude
15,A,L_125,minimally,2.550
0,A,A_13,minimally,2.366
6,A,L_106,minimally,2.221
10,A,L_113,minimally,2.220
5,A,L_105,minimally,1.425
13,A,V_121,minimally,1.030
7,A,V_107,minimally,1.030
12,A,F_117,minimally,1.009
3,A,Y_24,minimally,0.948
4,A,A_28,minimally,0.841


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Single Residue Frustration

In [ ]:
# @title Frustratometer in Python {"display-mode":"form"}
mode = "singleresidue" # @param ["configurational", "singleresidue", "mutational"]
pdbs_dir = "/content" # @param {type:"string"}
results_dir = "/content/Results" # @param {type:"string"}
example = True # @param {type:"boolean"}
overwrite = False # @param {type:"boolean"}
debug = False # @param {type:"boolean"}

import sys
import os
import frustrapy
import pickle

# Define residues to analyze
residues_to_analyze = {"A": [18, 19]}

# If the example is True, we will download the example files
if example:
    !wget -q http://www.rcsb.org/pdb/files/1fhj.pdb -O 1fhj.pdb
    !wget -q http://www.rcsb.org/pdb/files/2dn1.pdb -O 2dn1.pdb
    !wget -q http://www.rcsb.org/pdb/files/1m6k.pdb -O 1m6k.pdb

    pdbs_dir = "/content"
    results_dir = "/content/Results_example"
    # Remove any previous results
    !rm -rf /content/Results_example/*

if overwrite:
    if example:
        !rm -rf /content/Results/*
    else:
        # Convert the results_dir to an absolute path
        results_dir = os.path.abspath(results_dir)
        os.system(f"rm -rf {results_dir}/*")

# Directory frustration analysis with specific residues
plots_dir_dict = frustrapy.dir_frustration(
    pdbs_dir=pdbs_dir,
    mode=mode,
    results_dir=results_dir,
    debug=debug,
    chain="A",
    residues=residues_to_analyze
)

# Analyze and display results
results_found = 0

for root, dirs, files in os.walk(results_dir):
    for file in files:
        if file.endswith("_single_residue_data.pkl"):
            pkl_path = os.path.join(root, file)
            with open(pkl_path, "rb") as f:
                data = pickle.load(f)
            print(f"\nAnalysis results from: {os.path.basename(pkl_path)}")
            results_found += 1

            if "A" in data:
                for res_num in [18, 19]:
                    if res_num in data["A"]:
                        res_data = data["A"][res_num]
                        mutations = res_data.mutations

                        # Find most and least frustrated mutations
                        most_frustrated = min(mutations.items(), key=lambda x: x[1])
                        least_frustrated = max(mutations.items(), key=lambda x: x[1])

                        print(f"\nPosition {res_num} (Native: {res_data.residue_name})")
                        print(f"Most frustrated mutation: {res_data.residue_name} → {most_frustrated[0]} "
                              f"(Frustration Index: {most_frustrated[1]:.3f})")
                        print(f"Least frustrated mutation: {res_data.residue_name} → {least_frustrated[0]} "
                              f"(Frustration Index: {least_frustrated[1]:.3f})")

                        # Sort and display mutations
                        sorted_mutations = sorted(mutations.items(), key=lambda x: x[1])
                        print("\nAll mutations sorted by frustration (top 5 most and least frustrated):")
                        print("Most frustrated:")
                        for mut, score in sorted_mutations[:5]:
                            print(f"  {res_data.residue_name} → {mut}: {score:.3f}")
                        print("Least frustrated:")
                        for mut, score in sorted_mutations[-5:]:
                            print(f"  {res_data.residue_name} → {mut}: {score:.3f}")
                        print("-" * 50)

In [ ]:
# Show the fig objects
for pdb in plots_dir_dict.keys():
    for plot in plots_dir_dict[pdb].keys():
        fig = plots_dir_dict[pdb][plot]
        fig.show()

## Types of frustration modes the Frustratometer you can calculate in this notebook:

| Frustration Mode | Description | Mathematical Formula | Example Calculation |
|------------------|-------------|----------------------|---------------------|
| Configurational  | Compares the native energy of each contact in the protein to a set of decoy energies from random variants of the same contact. A contact is considered frustrated if its native energy is higher than the average of the decoys. | $F_c = \frac{E_n - \langle E_d \rangle}{\sigma_d}$ <br><br> $E_n$ = native energy of contact <br> $\langle E_d \rangle$ = mean energy of decoys <br> $\sigma_d$ = standard deviation of decoy energies | Native contact energy $E_n = -2.5$ <br> Mean decoy energy $\langle E_d \rangle = -5.2$ <br> Decoy std dev $\sigma_d = 1.8$ <br><br> $F_c = \frac{-2.5 - (-5.2)}{1.8} = 1.5$ <br><br> $F_c > 0$, so contact is frustrated |
| Mutational       | Compares the native energy of each contact to the average energy of all possible mutations of the amino acids forming that contact. A contact is considered frustrated if mutating it makes the energy more favorable on average. | $F_m = \frac{E_n - \langle E_m \rangle}{\sigma_m}$ <br><br> $E_n$ = native energy of contact <br> $\langle E_m \rangle$ = mean energy of all mutations <br> $\sigma_m$ = standard deviation of mutation energies | Native contact energy $E_n = -4.2$ <br> Mean mutation energy $\langle E_m \rangle = -6.8$ <br> Mutation std dev $\sigma_m = 2.1$ <br><br> $F_m = \frac{-4.2 - (-6.8)}{2.1} = 1.2$ <br><br> $F_m > 0$, so contact is frustrated |  
| Single Residue   | Calculates the total frustration of all contacts a single residue is involved in. Residues with many frustrated contacts are considered highly frustrated. | $F_r = \frac{1}{N} \sum_{i=1}^N F_{c,i}$ <br><br> $F_{c,i}$ = configurational frustration of $i$th contact <br> $N$ = total number of contacts residue is involved in | Residue involved in 3 contacts: <br> $F_{c,1} = 0.8$ <br> $F_{c,2} = 1.2$ <br> $F_{c,3} = -0.5$ <br><br> $F_r = \frac{1}{3}(0.8 + 1.2 + -0.5) = 0.5$ <br><br> $F_r > 0$, so residue is net frustrated |

In a nuthshell:
- Configurational frustration compares native contact energy to decoys
- Mutational frustration compares native contact energy to average mutation energy  
- Single residue frustration averages configurational frustration over all of a residue's contacts

The key equations are:

$F_c = \frac{E_n - \langle E_d \rangle}{\sigma_d}$ (configurational)

$F_m = \frac{E_n - \langle E_m \rangle}{\sigma_m}$ (mutational)  

$F_r = \frac{1}{N} \sum_{i=1}^N F_{c,i}$ (single residue)

Where $E_n$ is the native energy, $\langle E_d \rangle$ and $\langle E_m \rangle$ are mean decoy and mutation energies, and $\sigma_d$ and $\sigma_m$ are the standard deviations of the decoy and mutation energy distributions.

